In [ ]:
import kagglehub
import pandas as pd
import numpy as np
# Download latest version
# path = kagglehub.dataset_download("rohanrao/formula-1-world-championship-1950-2020")

# print("Path to dataset files:", path)

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from dotenv import load_dotenv
load_dotenv()

import getpass
import os

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your Groq API key: ")

In [ ]:
import sqlite3

conn = sqlite3.connect("f1_data.db")

csv_files = ["circuits.csv","constructor_results.csv","constructor_standings.csv",
             "drivers.csv", "races.csv", "results.csv", "constructors.csv",
             "driver_standings.csv",'drivers.csv','lap_times.csv',
             "pit_stops.csv","qualifying.csv","races.csv",'results.csv','seasons.csv',
             "sprint_results.csv",'status.csv']
csv_files_path = ["data" +"\\"+ c for c in csv_files]
for file in csv_files_path:
    df = pd.read_csv(file)
    # Clean column names (remove spaces/special chars for SQL safety)
    df.columns = [c.replace(' ', '_').lower() for c in df.columns]
    table_name = file.replace(".csv", "")
    df.to_sql(table_name, conn, if_exists="replace", index=False)

# Version 1: Multi-table SQLite LangGraph pipeline

Architecture:

User input -> Generate SQL query -> Fetch data from SQLite -> Summarize result -> Return answer

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    max_tokens=None,
    reasoning_format="parsed",
    timeout=None,
    max_retries=2
)

In [ ]:
class State(TypedDict):
    user_query:str
    sql_code:str
    sql_code_answer:str
    summary:str

Define the graph state and node functions for the V1 pipeline.

In [ ]:
# V1 uses plain LangGraph node functions, not LangChain tool wrappers.

The nodes below generate SQL, execute it against the local SQLite database, and summarize the result.

In [ ]:
conn = sqlite3.connect('f1_data.db')
cursor = conn.cursor()


table_names = [c.replace(".csv","") for c in csv_files]
schema_info_all = []
for table in table_names:
    cursor.execute(f"PRAGMA table_info({table})")

    columns = cursor.fetchall()
    schema_info_all.append([[col[1],col[2]] for col in columns])

table_schema_info={}
for i in range(len(table_names)):
        table_schema_info[table_names[i]] = schema_info_all[i]


def query_generator(state:State):
    """Function for transforming user input into SQL query"""
    prompt = f"""Generate SQL code for this query - {state['user_query']}, using the provided context in a dictionary: 
    where the keys are table_names and the values are the schema in a list form.
    {table_schema_info}. Respond only with the SQL query."""
    output = llm.invoke(prompt)
    return {'sql_code':output.content}


def run_sqlite_query(state:State):
    """
    Executes a SQL query against the F1 dataset and returns the results.
    Use this to join tables or aggregate F1 statistics.
    """
    conn = sqlite3.connect("f1_data.db")
    try:
        cursor = conn.cursor()
        cursor.execute(state['sql_code'])
        result = cursor.fetchall()
        return {'sql_code_answer':result}
    except Exception as e:
        return {'sql_code_answer': f"Error: {str(e)}"}
    finally:
        conn.close()
        


def summarizer(state:State):
    """ Provides a proper summary for the SQL query and its answer for the user."""
    prompt = f"""SYSTEM INSTRUCTION:
            You are an F1 Data Analyst. Your task is to take a user's question and the 
            corresponding SQL data results to provide a polished, expert response.

            INPUT DATA:
            1. User Question: {state['user_query']}
            2. Executed SQL: {state['sql_code']}
            3. SQL Result: {state['sql_code_answer']}

            GUIDELINES:
            - Directness: Answer the user's question in the first sentence.
            - Accuracy: Use ONLY the data provided in the 'SQL Result'. Do not hallucinate external stats.
            - Professionalism: If the result is a list of names or numbers, format them clearly (e.g., use bullet points for multiple drivers).
            - Context: If the SQL result is empty, politely explain that no records were found for that specific query in the F1 database.
            - Detail: Mention the specific driver names and values exactly as they appear in the result.

            RESPONSE FORMAT:
            "Based on the race data, [Your natural language answer here]."
            """
    output = llm.invoke(prompt)
    return {'summary':output.content}


In [ ]:
builder = StateGraph(State)
builder.add_node("query generator",query_generator)
builder.add_node("sqlite3 generator",run_sqlite_query)
builder.add_node("summarization",summarizer)
builder.set_entry_point("query generator")
builder.add_edge("query generator", "sqlite3 generator")
builder.add_edge("sqlite3 generator","summarization")
builder.add_edge("summarization",END)
graph = builder.compile()

result = graph.invoke({"user_query":"tell me who had the most DNFs in 2023?"})

In [ ]:
result

# Version 2:

1. Adding a routing decision

    - Setting up multiple subagents.
        - One for factual lookup
        - One for comparative/deeper analysis (over a season or multiple seasons)
        - Hybrid (lookup/analysis + structured summary)

2. Improved the system prompt for the LLM
3. Added PydanticAI (structured output) to ensure LLM always returns answers in one format. 

In [88]:
class State(TypedDict):
    user_query:str
    intent:str
    sql_code:str
    sql_code_answer:str
    summary:str

In [89]:
from pydantic import BaseModel, Field
from typing import Literal

In [90]:
from ast import List

class intent_classification(BaseModel):
    user_query: str = Field(description="The user query")
    intent: Literal["Factual Lookup","Deep Analysis","Hybrid"]


def intent_classifier(state:State):
    prompt = f"""You are an F1 expert. You have access to a SQLite database containing information
    about all the races from 1950 to 2024. Your task is to classify user question's intent using the schema below
    Schema Context = {table_schema_info}
    User Question = {state['user_query']}
    Intent: "Factual Lookup" or "Deep Analysis" or "Hybrid"
    ########################################
    
    RULES
    1. For each "User Question", classify it as one of three types:
        - Factual Lookup: Use this if the question has a single, verifiable answer found in the data (e.g., "What was the date of...", "Who won...").
        - Deep Analysis: Use this if the user wants an interpretation, a trend, or a statistical breakdown (e.g., "Why did...", "Analyze the trend...", "Was X better than Y?").
        - Hybrid: Use this if the user asks for a specific fact AND an analysis of that fact in one go.
    2. Always respond with one of the three possible Intents.

    """
    structuredllm = llm.with_structured_output(intent_classification,method='json_schema')
    response = structuredllm.invoke(prompt)
    return {'user_query':response.user_query,
            'intent':response.intent
            }

In [91]:
class factual_lookup_response(BaseModel):
    sql_code: str = Field(description="The generated SQL query")


def factual_lookup(state:State):
    prompt = f"""  
            Given a user question and a schema containing F1 races information, generate a SQL query to answer 
            the user's question.
            Schema Info: {table_schema_info}
            User Question: {state['user_query']}

            RULES:
            2. Ensure your SQL query is grounded in the schema and is valid and accurate.
            3. Validate the SQL query before answering.

        """
    structuredllm = llm.with_structured_output(factual_lookup_response,method='json_schema')
    response = structuredllm.invoke(prompt)
    return {'sql_code':response.sql_code}

In [92]:
class deep_analysis_response(BaseModel):
    sql_code: str = Field(description="The generated SQL query")


def deep_analysis(state:State):
    prompt = f"""  
            Given a user question and a schema containing F1 races information, generate a SQL query to answer 
            the user's question.
            Schema Info: {table_schema_info}
            User Question: {state['user_query']}

            RULES:
            1. Ensure your SQL query is grounded in the schema and is valid and accurate.
            2. Validate the SQL query before answering.

        """
    structuredllm = llm.with_structured_output(factual_lookup_response,method='json_schema')
    response = structuredllm.invoke(prompt)
    return {'sql_code':response.sql_code}

In [93]:
def hybrid_response(state:State):
    prompt = f""" Given a user question and the supporting schema/context behind it, generate a 
    SQL query to answer the question.
    User Question: {state['user_query']}
    Schema: {table_schema_info}
    
    RULES:
    1. Respond only with the SQL query, nothing else.
    2. The SQL query must always be grounded in the schema provided to you, not your general knowledge. 
            
    """
    response = llm.invoke(prompt)
    return {'sql_code':response.content}


In [94]:
def run_sqlite_query(state:State):
    """
    Executes a SQL query against the F1 dataset and returns the results.
    Use this to join tables or aggregate F1 statistics.
    """
    conn = sqlite3.connect("f1_data.db")
    try:
        cursor = conn.cursor()
        cursor.execute(state['sql_code'])
        result = cursor.fetchall()
        return {'sql_code_answer':result}
    except Exception as e:
        return {'sql_code_answer': f"Error: {str(e)}"}
    finally:
        conn.close()
        

In [127]:
def polished_response(state:State):
        prompt =  f"""
  SYSTEM INSTRUCTION:
  You are an F1 Data Analyst. Convert the provided SQL result into a clear, polished response.

  INPUTS:
  - User Question: {state['user_query']}
  - Intent: {state['intent']}                                                                                                        
  - SQL query created based on the User Question: {state['sql_code']}                                                                
  - SQL query answer: {state['sql_code_answer']}                                                                                     
                                                                                                                                     
  GUIDELINES:                                                                                                                        
  1. Use ONLY the SQL query answer as the source of truth.                                                                           
  2. Do not invent race names, teammate names, team names, causes, or comparisons unless they are present in the SQL result.         
  3. Do not reproduce large SQL result tables in full.                                                                               
  4. If the SQL result has more than 8 rows, summarize it instead of printing every row.                                             
  5. For race-by-race results, show only:                                                                                            
     - best results                                                                                                                  
     - worst results                                                                                                                 
     - notable patterns                                                                                                              
     - aggregate totals                                                                                                              
  6. Use small Markdown tables only. Maximum 6 rows per table.                                                                       
  7. Keep paragraphs short.                                                                                                          
  8. You may calculate simple derived values from the SQL result, such as percentages or averages.  
  9. Use plain ASCII spaces only. Do not use Unicode spaces such as \\u202f or non-breaking spaces.                                  
  10. Do not use special typography characters. Use normal apostrophes, hyphens, and spaces.                                 
  11. Keep the response aligned with the Intent:                                                                                      
     - If Intent='Factual Lookup', answer in 1-2 concise sentences.                                                                  
     - If Intent='Deep Analysis', provide a compact analytical summary.                                                              
     - If Intent='Hybrid', start with the factual answer, then add brief analysis.                                                   
                                                                                                                                     
  RESPONSE FORMAT FOR DEEP ANALYSIS:                                                                                                 
  ## Summary                                                                                                                         
  One direct paragraph of 2-3 sentences.                                                                                             
                                                                                                                                     
  ## Key Metrics                                                                                                                     
  A compact Markdown table with at most 6 rows.
                                                                                                                                     
  ## Highlights                                                                                                                      
  3-5 bullets summarizing the most important patterns.                                                                               
                                                                                                                                     
  ## Caveat                                                                                                                          
  One sentence explaining what this SQL result does not cover.                                                                       
                                                                                                                                     
  Your response must ONLY contain the polished response.                                                                             
  """                           
        response = llm.invoke(prompt)
        return {'summary':response.content}


In [128]:
def routing_decision(state:State):
    if state['intent']=='Factual Lookup':
        return 0
    elif state['intent']=='Deep Analysis':
        return 1
    else:
        return 2

In [129]:
builder = StateGraph(State)

builder.add_node("intent classifier",intent_classifier)
builder.add_node("sqlite3 generator",run_sqlite_query)
builder.add_node("summarization",polished_response)
builder.add_node('factual lookup',factual_lookup)
builder.add_node('deep analysis',deep_analysis)
builder.add_node('hybrid',hybrid_response)

builder.add_conditional_edges(
    'intent classifier',
    routing_decision,
    {
        0:'factual lookup',
        1:'deep analysis',
        2:'hybrid'
    }
)
builder.add_edge('factual lookup','sqlite3 generator')
builder.add_edge('deep analysis','sqlite3 generator')
builder.add_edge('hybrid','sqlite3 generator')
builder.add_edge('sqlite3 generator','summarization')
builder.set_entry_point('intent classifier')
builder.add_edge('summarization',END)

graph = builder.compile()

result = graph.invoke({"user_query":"analyze norris's performance in 2024 season"})

In [130]:
result

{'user_query': "analyze norris's performance in 2024 season",
 'intent': 'Deep Analysis',
 'sql_code': "SELECT r.year,\n       r.round,\n       r.name AS race_name,\n       c.name AS constructor,\n       res.position,\n       res.points,\n       st.status AS finish_status\nFROM races r\nJOIN results res ON r.raceid = res.raceid\nJOIN drivers d ON res.driverid = d.driverid\nJOIN constructors c ON res.constructorid = c.constructorid\nJOIN status st ON res.statusid = st.statusid\nWHERE d.driverref = 'norris'\n  AND r.year = 2024\nORDER BY r.round;",
 'sql_code_answer': [(2024,
   1,
   'Bahrain Grand Prix',
   'McLaren',
   '6',
   8.0,
   'Finished'),
  (2024, 2, 'Saudi Arabian Grand Prix', 'McLaren', '8', 4.0, 'Finished'),
  (2024, 3, 'Australian Grand Prix', 'McLaren', '3', 15.0, 'Finished'),
  (2024, 4, 'Japanese Grand Prix', 'McLaren', '5', 10.0, 'Finished'),
  (2024, 5, 'Chinese Grand Prix', 'McLaren', '2', 18.0, 'Finished'),
  (2024, 6, 'Miami Grand Prix', 'McLaren', '1', 25.0, 'Fi